In [16]:
import numpy as np
import random
import pandas as pd
import numpy_financial as npf

In [17]:
class cliente:
    def __init__(self, codigo):
        self.codigo = codigo
        self.inad = max(0.06 + np.random.normal(0., 0.005),0.003)
        self.prop = np.random.uniform(low=0.0, high=1.0, size=None)

## Simula base de clientes

In [18]:
clientes = []

for i in range(1000000):
    clientes.append(cliente(str(i).zfill(7)))

In [19]:
class operacao:
    def __init__(self, contrato, ref,  mes, dia):
        self.ref = ref
        self.valor = np.random.lognormal(6., 1.8, size=None) + 5000
        self.prazo = int(np.random.uniform(low=1, high=36, size=None))
        self.juros = 0.01
        self.parcela = npf.pmt(self.juros, self.prazo, self.valor)
        self.multa = 0
        self.pagamento = 0
        self.contrato = contrato
        self.cliente = random.sample(clientes, 1)[0]
        self.atraso = 0
        self.mes = mes
        self.dia = dia

### Gerar Operações por dia e agregar na base com o mês

In [20]:
def contrata(mes, ctr_dia, ref):
    
    operacoes = []

    for d in range(1, 31):
        q = np.random.poisson(lam=ctr_dia, size=None)
        
        for i in range(q):
            operacoes.append(operacao(str(mes).zfill(3) + str(d).zfill(2)+(str(i)).zfill(6), ref, mes, d))
            
    return(operacoes)

### Atualização do mês
#### e inclusão de novas contratações

In [21]:
def atualiza_mes(anterior, ctr_dia=500):
    #print(len(anterior))
    atual = []

    for i in range(len(anterior)):

        atual.append(anterior[i])

        dia = 30 - anterior[i].dia
        
        #ajustar atraso para somar 30 dias se menor que 30 dia se não + 30
        
        p_atraso = np.random.binomial(size = None, n = 1, p = anterior[i].cliente.inad)
        
        atual[i].atraso = p_atraso * anterior[i].atraso + p_atraso * dia

        if anterior[i].atraso < 30:
            p = 0.06 + 0.24 * (atual[i].atraso - 0)/30
        elif anterior[i].atraso < 60:
            p = 0.3 + 0.3 * (atual[i].atraso - 30)/30
        elif anterior[i].atraso < 90:    
            p = 0.6 + 0.34 * (atual[i].atraso - 60)/30
        elif anterior[i].atraso < 120:
            p = 0.94
        elif anterior[i].atraso < 150:
            p = 0.98
        elif anterior[i].atraso < 180:
            p = 0.993  
        elif anterior[0].atraso < 270:
            p = 0.995
        elif anterior[i].atraso < 300:
            p = 0.997
        elif anterior[i].atraso < 330:
            p = 0.999  
        elif anterior[i].atraso < 360:
            p = 0.9995
        else:
            p = 1

        atual[i].cliente.inad = p
        atual[i].prazo = anterior[i].prazo - 1
        atual[i].ref = anterior[i].ref + 1
        atual[i].pagamento = (-anterior[i].parcela + anterior[i].multa) * (1 - p_atraso) 
        atual[i].valor = max((anterior[i].valor) * (1 + anterior[i].juros) + ((anterior[i].parcela - atual[i].multa) * (1 - p_atraso)), 0)

        atual[i].multa = - (anterior[i].parcela * p_atraso * (1 + anterior[i].juros)) + (anterior[i].multa * p_atraso)
        
        
    atual.extend(contrata(anterior[0].mes + 1, ctr_dia, anterior[0].ref))
    
    return(atual)

        ## Incluir probabilidade de inadimplência em 1 mês ok
        ## atualizar dias de atraso ok
        ## utilizar probabilidades de recuperação e inadimplência do System Dynamics ok
        
        
        ## atualizar mes de referencia ok
        ## atualizar prazo ok
        ## atualizar liquidação ok
        ## exportar cada nova tabela ok
        ## criar loop ok
        ## atualizar probabilidades na base de clientes ok

## Bases geradas

In [22]:
base0 = contrata(0, 500, 0)

data = {'ref':[], 'cliente':[], 'contrato':[], 'prazo':[], 'valor':[], 'mes':[], 'dia':[], 'prob':[],'atraso':[], 'pgto':[]}

df = pd.DataFrame(data)

for n in range(len(base0)):
    df.loc[len(df.index)] = [base0[n].ref , base0[n].cliente.codigo, base0[n].contrato, base0[n].prazo, base0[n].valor, base0[n].mes, base0[n].dia, base0[n].cliente.inad, base0[n].atraso, base0[n].pagamento]

df.to_csv(path_or_buf='Simulacao_mes_0.csv', sep=';',header=True, decimal=',', index=False)

print("Simulacao_mes_0.csv gravada com sucesso.")

for i in range(1):
    
    exec("base" + str(i+1) + "= atualiza_mes(" + "base" + str(i) + ")")
    
    data = {'ref':[], 'cliente':[], 'contrato':[], 'prazo':[], 'valor':[], 'mes':[], 'dia':[], 'prob':[],'atraso':[], 'pgto':[]}
    
    exec("df" + str(i) + "= pd.DataFrame(data)")
    
    exec("linhas = len(base" + str(i + 1) + ")")
    
    for n in range(linhas):
        exec("df" + str(i) + ".loc[len(df" + str(i) + ".index)] = [" + "base" + str(i+1) + "[n].ref , base" 
             + str(i+1) + "[n].cliente.codigo, base" + str(i+1) + "[n].contrato, base" + str(i+1) + "[n].prazo, base"+
             str(i+1) + "[n].valor, base" + str(i+1) + "[n].mes, base" + str(i+1) + "[n].dia, base" + 
             str(i+1) + "[n].cliente.inad, base" + str(i + 1) + "[n].atraso, base" + str(i+1) + "[n].pagamento]")
        
    exec("df" + str(i) + ".to_csv(path_or_buf='Simulacao_mes_" + str(i+1) + ".csv', sep=';',header=True, decimal=',', index=False)")
    print("Simulacao_mes_" + str(i + 1) + ".csv gravada com sucesso.")

KeyboardInterrupt: 

In [8]:
base0 = contrata(0, 1, 0)

In [9]:
for i in range(len(base0)):
    print("ref:", base0[i].ref,
          "contrato:",  base0[i].contrato,
          "prazo:", base0[i].prazo,
          "atraso:", base0[i].atraso, 
          "pagamento:", "%6.2f"% base0[i].pagamento,
          "parcela:",  "%6.2f"% base0[i].parcela,
          "multa:",  "%6.2f"% base0[i].multa,
          "valor:" , "%6.2f"% base0[i].valor)  

ref: 0 contrato: 00001000000 prazo: 11 atraso: 0 pagamento:   0.00 parcela: -527.27 multa:   0.00 valor: 5466.52
ref: 0 contrato: 00001000001 prazo: 11 atraso: 0 pagamento:   0.00 parcela: -570.93 multa:   0.00 valor: 5919.22
ref: 0 contrato: 00001000002 prazo: 3 atraso: 0 pagamento:   0.00 parcela: -1752.11 multa:   0.00 valor: 5152.92
ref: 0 contrato: 00001000003 prazo: 28 atraso: 0 pagamento:   0.00 parcela: -207.04 multa:   0.00 valor: 5034.41
ref: 0 contrato: 00002000000 prazo: 21 atraso: 0 pagamento:   0.00 parcela: -512.24 multa:   0.00 valor: 9659.29
ref: 0 contrato: 00002000001 prazo: 6 atraso: 0 pagamento:   0.00 parcela: -870.32 multa:   0.00 valor: 5043.89
ref: 0 contrato: 00002000002 prazo: 30 atraso: 0 pagamento:   0.00 parcela: -305.23 multa:   0.00 valor: 7877.31
ref: 0 contrato: 00003000000 prazo: 33 atraso: 0 pagamento:   0.00 parcela: -189.18 multa:   0.00 valor: 5295.19
ref: 0 contrato: 00003000001 prazo: 4 atraso: 0 pagamento:   0.00 parcela: -2463.18 multa:   0.00

In [10]:
base1 = atualiza_mes(base0, 1)

In [11]:
for i in range(len(base0)):
    print("ref:", base1[i].ref,
          "contrato:",  base1[i].contrato,
          "prazo:", base1[i].prazo,
          "atraso:", base1[i].atraso, 
          "pagamento:", "%6.2f"% base1[i].pagamento,
          "parcela:",  "%6.2f"% base1[i].parcela,
          "multa:",  "%6.2f"% base1[i].multa,
          "valor:" , "%6.2f"% base1[i].valor)   

ref: 1 contrato: 00001000000 prazo: 10 atraso: 0 pagamento: 527.27 parcela: -527.27 multa:   0.00 valor: 4993.92
ref: 1 contrato: 00001000001 prazo: 10 atraso: 29 pagamento:   0.00 parcela: -570.93 multa: 576.64 valor: 5978.41
ref: 1 contrato: 00001000002 prazo: 2 atraso: 0 pagamento: 1752.11 parcela: -1752.11 multa:   0.00 valor: 3452.35
ref: 1 contrato: 00001000003 prazo: 27 atraso: 0 pagamento: 207.04 parcela: -207.04 multa:   0.00 valor: 4877.71
ref: 1 contrato: 00002000000 prazo: 20 atraso: 0 pagamento: 512.24 parcela: -512.24 multa:   0.00 valor: 9243.64
ref: 1 contrato: 00002000001 prazo: 5 atraso: 0 pagamento: 870.32 parcela: -870.32 multa:   0.00 valor: 4224.02
ref: 1 contrato: 00002000002 prazo: 29 atraso: 0 pagamento: 305.23 parcela: -305.23 multa:   0.00 valor: 7650.85
ref: 1 contrato: 00003000000 prazo: 32 atraso: 0 pagamento: 189.18 parcela: -189.18 multa:   0.00 valor: 5158.96
ref: 1 contrato: 00003000001 prazo: 3 atraso: 0 pagamento: 2463.18 parcela: -2463.18 multa:   0

In [12]:
base2 = atualiza_mes(base1)

In [13]:
for i in range(len(base0)):
    print("ref:", base2[i].ref,
          "contrato:",  base2[i].contrato,
          "prazo:", base2[i].prazo,
          "atraso:", base2[i].atraso, 
          "pagamento:", "%6.2f"% base2[i].pagamento,
          "parcela:",  "%6.2f"% base2[i].parcela,
          "multa:",  "%6.2f"% base2[i].multa,
          "valor:" , "%6.2f"% base2[i].valor)   

ref: 2 contrato: 00001000000 prazo: 9 atraso: 0 pagamento: 527.27 parcela: -527.27 multa:   0.00 valor: 4516.59
ref: 2 contrato: 00001000001 prazo: 9 atraso: 0 pagamento: 1147.57 parcela: -570.93 multa:   0.00 valor: 4890.62
ref: 2 contrato: 00001000002 prazo: 1 atraso: 0 pagamento: 1752.11 parcela: -1752.11 multa:   0.00 valor: 1734.76
ref: 2 contrato: 00001000003 prazo: 26 atraso: 0 pagamento: 207.04 parcela: -207.04 multa:   0.00 valor: 4719.45
ref: 2 contrato: 00002000000 prazo: 19 atraso: 0 pagamento: 512.24 parcela: -512.24 multa:   0.00 valor: 8823.84
ref: 2 contrato: 00002000001 prazo: 4 atraso: 0 pagamento: 870.32 parcela: -870.32 multa:   0.00 valor: 3395.94
ref: 2 contrato: 00002000002 prazo: 28 atraso: 0 pagamento: 305.23 parcela: -305.23 multa:   0.00 valor: 7422.13
ref: 2 contrato: 00003000000 prazo: 31 atraso: 0 pagamento: 189.18 parcela: -189.18 multa:   0.00 valor: 5021.36
ref: 2 contrato: 00003000001 prazo: 2 atraso: 0 pagamento: 2463.18 parcela: -2463.18 multa:   0.0

In [14]:
base3 = atualiza_mes(base2)

In [15]:
for i in range(len(base0)):
    print("ref:", base3[i].ref,
          "contrato:",  base3[i].contrato,
          "prazo:", base3[i].prazo,
          "atraso:", base3[i].atraso, 
          "pagamento:", "%6.2f"% base3[i].pagamento,
          "parcela:",  "%6.2f"% base3[i].parcela,
          "multa:",  "%6.2f"% base3[i].multa,
          "valor:" , "%6.2f"% base3[i].valor) 

ref: 3 contrato: 00001000000 prazo: 8 atraso: 0 pagamento: 527.27 parcela: -527.27 multa:   0.00 valor: 4034.49
ref: 3 contrato: 00001000001 prazo: 8 atraso: 29 pagamento:   0.00 parcela: -570.93 multa: 576.64 valor: 4939.53
ref: 3 contrato: 00001000002 prazo: 0 atraso: 0 pagamento: 1752.11 parcela: -1752.11 multa:   0.00 valor:   0.00
ref: 3 contrato: 00001000003 prazo: 25 atraso: 29 pagamento:   0.00 parcela: -207.04 multa: 209.11 valor: 4766.65
ref: 3 contrato: 00002000000 prazo: 18 atraso: 0 pagamento: 512.24 parcela: -512.24 multa:   0.00 valor: 8399.84
ref: 3 contrato: 00002000001 prazo: 3 atraso: 0 pagamento: 870.32 parcela: -870.32 multa:   0.00 valor: 2559.59
ref: 3 contrato: 00002000002 prazo: 27 atraso: 0 pagamento: 305.23 parcela: -305.23 multa:   0.00 valor: 7191.12
ref: 3 contrato: 00003000000 prazo: 30 atraso: 0 pagamento: 189.18 parcela: -189.18 multa:   0.00 valor: 4882.39
ref: 3 contrato: 00003000001 prazo: 1 atraso: 0 pagamento: 2463.18 parcela: -2463.18 multa:   0.0